# Trade Risk Assessment Ensemble

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

## Load Trade Risk Data

In [ ]:
data_paths = [
    'data/final_csv/04_trade_risk_eda.csv',
    '../data/final_csv/04_trade_risk_eda.csv',
    'backend/brain/data/final_csv/04_trade_risk_eda.csv',
    'brain/datasets/dump/brain_prev/data_pipeline/data/final_csv/04_trade_risk_eda.csv'
]
path = next(p for p in data_paths if os.path.exists(p))
df = pd.read_csv(path)
df.head()

## Risk Features Overview

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Risk features: {len(num_cols)}")
df[num_cols[:6]].describe()

## Model Ensemble: Isolation Forest + GRU Autoencoder

In [ ]:
# GRU sequence autoencoder
class GRUAutoencoder(nn.Module):
    def __init__(self, input_dim=27, hidden_dim=32, latent_dim=16):
        super().__init__()
        self.encoder = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc_enc = nn.Linear(hidden_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, h = self.encoder(x)
        latent = torch.relu(self.fc_enc(h[-1]))
        dec_init = torch.relu(self.fc_dec(latent)).unsqueeze(0)
        out, _ = self.decoder(torch.zeros_like(x), dec_init)
        return self.output_layer(out)

# isolation forest for point anomalies
X_sample = df[num_cols[:10]].fillna(0).values
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_sample)

iso = IsolationForest(n_estimators=100, random_state=42)
iso.fit(X_scaled)
df['point_anomaly_score'] = -iso.decision_function(X_scaled)

## Composite Risk Scoring

In [ ]:
df['risk_score'] = (df['point_anomaly_score'] - df['point_anomaly_score'].min()) / (df['point_anomaly_score'].max() - df['point_anomaly_score'].min() + 1e-6) * 100
df['risk_level'] = pd.cut(df['risk_score'], bins=[-1, 30, 70, 100], labels=['LOW', 'MEDIUM', 'HIGH'])

print(df['risk_level'].value_counts())
display_cols = [c for c in ['partner_iso3', 'hs6', 'risk_score', 'risk_level'] if c in df.columns]
df[display_cols].head(10)